# KoGPT2 문장 생성 실습 - PyTorch 개선 완성본

이 노트북은 Google Colab에서 KoGPT2 문장 생성 결과가 깨지거나, 시작 문장과 의미가 잘 이어지지 않는 문제를 줄이도록 수정한 PyTorch 버전입니다.

기존 문제의 주요 원인은 다음과 같습니다.

1. `max_length`를 크게 주면 입력 문장 이후 너무 긴 토큰을 계속 생성하면서 문맥이 흐트러질 수 있습니다.
2. 탐욕적 생성 방식은 매 단계에서 점수가 가장 높은 토큰만 선택하므로 반복되거나 어색한 문장이 나올 수 있습니다.
3. `repetition_penalty`만 크게 주면 반복은 줄어들 수 있지만, 모델이 자연스럽지 않은 후보를 선택할 수 있습니다.
4. KoGPT2는 토큰 단위로 문장을 생성하므로 일부 특수 문자나 깨져 보이는 문자가 출력될 수 있습니다.

해결 방향은 다음과 같습니다.

- `max_length` 대신 `max_new_tokens`를 사용하여 새로 생성할 길이만 제한합니다.
- `do_sample=True`, `top_p`, `top_k`, `temperature`를 사용하여 자연스러운 후보 안에서 문장을 생성합니다.
- `no_repeat_ngram_size`로 같은 구절 반복을 줄입니다.
- `eos_token_id`, `pad_token_id`를 명확히 지정하여 생성 종료와 패딩 처리를 안정화합니다.
- 생성 후 불필요한 제어 문자와 깨진 문자를 정리하는 후처리 함수를 적용합니다.

## 1. 필요한 패키지 설치

KoGPT2 모델은 Hugging Face `transformers` 라이브러리에서 불러옵니다.  
PyTorch 모델을 사용하므로 `torch`도 필요합니다.  
Colab에는 보통 PyTorch가 기본 설치되어 있지만, 버전 문제를 줄이기 위해 `transformers`와 `sentencepiece`를 함께 설치합니다.

In [ ]:
# 현재 노트북 커널에 필요한 패키지를 설치합니다.
# - transformers: Hugging Face 사전 학습 모델과 토크나이저를 사용하기 위한 라이브러리입니다.
# - sentencepiece: 일부 토크나이저가 내부적으로 사용할 수 있는 서브워드 토큰화 라이브러리입니다.
# - accelerate: Colab GPU 환경에서 모델 로딩과 실행을 보조하는 라이브러리입니다.
%pip install -q transformers sentencepiece accelerate

## 2. 라이브러리 불러오기

PyTorch 기반 KoGPT2 문장 생성을 위해 `torch`, `PreTrainedTokenizerFast`, `GPT2LMHeadModel`을 사용합니다.  
또한 출력 문장 후처리를 위해 `re` 모듈을 사용합니다.

In [ ]:
# 정규표현식 처리를 위한 파이썬 기본 라이브러리입니다.
# 생성된 문장에서 제어 문자, 깨진 문자, 과도한 공백을 제거할 때 사용합니다.
import re

# 표준 출력 인코딩을 확인하고 필요할 때 재설정하기 위한 파이썬 기본 라이브러리입니다.
# 일부 실행 환경에서 한글 출력이 깨질 때 UTF-8 출력 설정을 보조합니다.
import sys

# PyTorch 라이브러리입니다.
# Tensor 생성, GPU 사용, 모델 추론, 난수 고정 등에 사용합니다.
import torch

# PreTrainedTokenizerFast는 Hugging Face에서 제공하는 빠른 토크나이저 클래스입니다.
# KoGPT2처럼 사전 학습된 GPT 계열 모델의 토큰화를 빠르게 수행할 수 있습니다.
from transformers import PreTrainedTokenizerFast

# GPT2LMHeadModel은 GPT2 구조에 다음 토큰 예측용 LM Head가 붙어 있는 모델 클래스입니다.
# KoGPT2는 GPT2 계열 구조이므로 이 클래스로 문장 생성 모델을 불러올 수 있습니다.
from transformers import GPT2LMHeadModel

# 현재 실행 환경이 표준 출력 인코딩 재설정을 지원하는지 확인합니다.
# Jupyter/Colab 환경에서는 보통 UTF-8이 기본이지만, 일부 환경에서는 명시 설정이 도움이 됩니다.
if hasattr(sys.stdout, "reconfigure"):
    # 표준 출력 인코딩을 UTF-8로 설정합니다.
    # 한글 문자열이 print()로 출력될 때 깨지는 문제를 줄이는 보조 설정입니다.
    sys.stdout.reconfigure(encoding="utf-8")


## 3. 실행 장치 설정

PyTorch에서는 모델과 입력 Tensor가 같은 장치에 있어야 합니다.  
GPU가 사용 가능하면 `cuda`, 그렇지 않으면 `cpu`를 사용합니다.

In [ ]:
# 현재 Colab 런타임에서 GPU를 사용할 수 있는지 확인합니다.
# GPU가 있으면 "cuda", 없으면 "cpu" 장치를 사용합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 실제 실행 장치를 출력하여 GPU 사용 여부를 확인합니다.
print("사용 장치:", device)

## 4. 한글 깨짐 방지를 위한 KoGPT2 토크나이저와 모델 불러오기

모델과 토크나이저는 반드시 같은 모델 이름으로 불러와야 합니다.  
토크나이저가 만든 정수 ID 체계와 모델의 어휘 사전이 서로 일치해야 하기 때문입니다.

여기서는 공개 KoGPT2 모델인 `skt/kogpt2-base-v2`를 사용합니다. 한글 출력 깨짐을 줄이기 위해 KoGPT2 전용 특수 토큰을 명시하여 `PreTrainedTokenizerFast`로 토크나이저를 불러옵니다.

In [ ]:
# 사용할 KoGPT2 모델 이름을 지정합니다.
# 같은 model_name으로 토크나이저와 모델을 불러와야 토큰 ID 체계가 일치합니다.
model_name = "skt/kogpt2-base-v2"

# KoGPT2 토크나이저를 PreTrainedTokenizerFast 방식으로 불러옵니다.
# 이 방식은 KoGPT2에서 사용하는 특수 토큰을 명시할 수 있어 한글 디코딩 안정성을 높이는 데 도움이 됩니다.
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    model_name,          # Hugging Face Hub에서 불러올 KoGPT2 모델 이름입니다.
    bos_token='</s>',    # 문장 시작 토큰입니다. KoGPT2에서는 </s>를 시작/종료 계열 토큰으로 사용합니다.
    eos_token='</s>',    # 문장 종료 토큰입니다. generate()가 문장 생성을 멈출 기준으로 사용할 수 있습니다.
    unk_token='<unk>',   # 토크나이저가 모르는 단어 또는 토큰을 만났을 때 사용할 토큰입니다.
    pad_token='<pad>',   # 길이가 다른 문장을 배치로 묶을 때 짧은 문장을 채우는 패딩 토큰입니다.
    mask_token='<mask>'  # 마스크 토큰입니다. GPT 생성에서는 주로 사용하지 않지만 토크나이저 설정 완성도를 위해 지정합니다.
)

# PyTorch 기반 KoGPT2 언어 모델을 GPT2LMHeadModel 방식으로 불러옵니다.
# GPT2LMHeadModel은 입력된 토큰 뒤에 이어질 다음 토큰을 예측하는 생성 모델입니다.
model = GPT2LMHeadModel.from_pretrained(model_name)

# 모델을 GPU 또는 CPU로 이동합니다.
# 입력 Tensor도 같은 device에 올려야 장치 불일치 오류 없이 실행됩니다.
model = model.to(device)

# 이 노트북에서는 추가 학습이 아니라 문장 생성 추론을 수행하므로 평가 모드로 전환합니다.
# eval()은 Dropout처럼 학습 단계에서만 사용하는 기능을 비활성화합니다.
model.eval()

# generate() 함수가 패딩 토큰을 정확히 인식하도록 모델 설정에 pad_token_id를 지정합니다.
# pad_token_id가 없으면 generate() 실행 중 경고가 나오거나 attention_mask 처리에 문제가 생길 수 있습니다.
model.config.pad_token_id = tokenizer.pad_token_id

# generate() 함수가 문장 종료 토큰을 정확히 인식하도록 모델 설정에 eos_token_id를 지정합니다.
# eos_token_id는 모델이 문장을 끝낼 시점을 판단하는 데 사용됩니다.
model.config.eos_token_id = tokenizer.eos_token_id

# generate() 함수가 문장 시작 토큰을 참조할 수 있도록 bos_token_id도 함께 지정합니다.
# bos_token_id는 문장 시작을 나타내는 특수 토큰 ID입니다.
model.config.bos_token_id = tokenizer.bos_token_id

# 로드 결과를 확인합니다.
# 토큰 문자열과 토큰 ID가 제대로 지정되었는지 출력하여 설정 상태를 점검합니다.
print("모델 로드 완료:", model_name)
print("bos_token:", tokenizer.bos_token, "| bos_token_id:", tokenizer.bos_token_id)
print("eos_token:", tokenizer.eos_token, "| eos_token_id:", tokenizer.eos_token_id)
print("unk_token:", tokenizer.unk_token, "| unk_token_id:", tokenizer.unk_token_id)
print("pad_token:", tokenizer.pad_token, "| pad_token_id:", tokenizer.pad_token_id)
print("mask_token:", tokenizer.mask_token, "| mask_token_id:", tokenizer.mask_token_id)


## 5. 재현 가능한 결과를 위한 난수 고정

샘플링 방식은 후보 토큰 중에서 확률적으로 토큰을 선택합니다.  
따라서 같은 코드를 실행해도 결과가 달라질 수 있습니다.  
실습에서는 결과 비교가 쉽도록 난수 시드를 고정합니다.

In [ ]:
# PyTorch CPU 연산의 난수 시드를 고정합니다.
# 같은 환경에서 실행하면 샘플링 결과가 어느 정도 재현됩니다.
torch.manual_seed(42)

# GPU를 사용하는 경우 GPU 연산의 난수 시드도 함께 고정합니다.
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# 재현성 설정 완료 메시지를 출력합니다.
print("난수 시드 고정 완료")

## 6. 문장 정리 함수 만들기

KoGPT2가 생성한 문장에는 간혹 제어 문자, 깨진 문자, 과도한 공백이 포함될 수 있습니다.  
아래 함수는 생성 결과를 사람이 읽기 좋게 정리합니다.

이 함수는 모델 성능을 바꾸는 것이 아니라 출력 문자열만 정리합니다.

In [ ]:
# 생성된 문자열을 정리하는 함수를 정의합니다.
# text 매개변수에는 tokenizer.decode()로 복원된 문자열이 들어옵니다.
def clean_generated_text(text: str) -> str:
    # 유니코드 replacement character(�)는 디코딩 과정에서 깨진 글자가 있을 때 나타날 수 있으므로 제거합니다.
    text = text.replace("�", "")

    # ASCII 제어 문자 범위를 공백으로 바꿉니다.
    # 줄바꿈, 탭, 보이지 않는 제어 문자가 섞이면 출력이 지저분해질 수 있습니다.
    text = re.sub(r"[\x00-\x1F\x7F]", " ", text)

    # KoGPT2 토크나이저에서 출력될 수 있는 특수 토큰 문자열을 제거합니다.
    # 문장 생성 결과에는 사람이 읽는 문장만 남기는 것이 좋습니다.
    text = text.replace("</s>", " ")
    text = text.replace("<pad>", " ")
    text = text.replace("<unk>", " ")
    text = text.replace("<mask>", " ")

    # 여러 개의 공백을 하나의 공백으로 줄입니다.
    # 토큰 디코딩 후 공백이 반복될 수 있으므로 보기 좋게 정리합니다.
    text = re.sub(r"\s+", " ", text)

    # 문장 앞뒤의 불필요한 공백을 제거합니다.
    # 최종 출력 문자열을 깔끔하게 만들기 위한 처리입니다.
    text = text.strip()

    # 정리된 문자열을 반환합니다.
    return text


## 7. 문장 생성 함수 만들기

문장 생성 품질을 개선하기 위해 다음 설정을 사용합니다.

- `max_new_tokens`: 입력 문장을 제외하고 새로 생성할 토큰 수만 제한합니다.
- `do_sample=True`: 항상 1등 토큰만 고르지 않고 확률적으로 후보를 선택합니다.
- `top_p`: 누적 확률이 일정 비율 안에 들어오는 후보만 사용합니다.
- `top_k`: 점수가 높은 상위 후보 일부만 사용합니다.
- `temperature`: 낮을수록 안정적이고, 높을수록 다양해집니다.
- `no_repeat_ngram_size`: 같은 n-gram 구절 반복을 줄입니다.
- `repetition_penalty`: 이미 나온 토큰의 반복을 완화합니다.

의미 연결성을 높이려면 `temperature`를 너무 높게 주지 않고, 시작 문장을 구체적으로 작성하는 것이 좋습니다.

In [ ]:
# KoGPT2로 문장을 생성하는 함수를 정의합니다.
# prompt는 사용자가 입력하는 시작 문장입니다.
def generate_korean_text(
    prompt: str,
    max_new_tokens: int = 60,
    temperature: float = 0.75,
    top_p: float = 0.90,
    top_k: int = 50,
    repetition_penalty: float = 1.15,
    no_repeat_ngram_size: int = 3,
) -> str:
    # 입력 문장을 토크나이저로 인코딩합니다.
    # return_tensors="pt"는 결과를 PyTorch Tensor 형태로 반환하라는 의미입니다.
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    )

    # 인코딩 결과 중 input_ids를 모델과 같은 device로 이동합니다.
    input_ids = encoded["input_ids"].to(device)

    # attention_mask가 있으면 모델과 같은 device로 이동합니다.
    # attention_mask는 실제 토큰과 패딩 토큰을 구분하는 역할을 합니다.
    attention_mask = encoded.get("attention_mask", None)
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)

    # 문장 생성은 학습이 아니므로 기울기 계산을 비활성화합니다.
    # 이렇게 하면 메모리 사용량이 줄고 실행 속도가 좋아집니다.
    with torch.no_grad():
        # generate() 함수로 입력 문장 뒤에 이어질 토큰을 생성합니다.
        generated_ids = model.generate(
            input_ids=input_ids,                         # 모델 입력 토큰 ID입니다.
            attention_mask=attention_mask,               # 패딩 여부를 알려주는 마스크입니다.
            max_new_tokens=max_new_tokens,               # 새로 생성할 토큰 수만 제한합니다.
            do_sample=True,                              # 확률적 샘플링으로 자연스러운 후보를 선택합니다.
            temperature=temperature,                     # 생성 다양성을 조절합니다.
            top_p=top_p,                                 # 누적 확률 기준으로 후보를 제한합니다.
            top_k=top_k,                                 # 상위 k개 후보로 선택 범위를 제한합니다.
            repetition_penalty=repetition_penalty,       # 같은 토큰 반복을 줄입니다.
            no_repeat_ngram_size=no_repeat_ngram_size,   # 같은 구절 반복을 줄입니다.
            eos_token_id=tokenizer.eos_token_id,         # 문장 종료 토큰 ID를 지정합니다.
            pad_token_id=tokenizer.pad_token_id,         # 패딩 토큰 ID를 지정합니다.
            use_cache=True                               # 이전 계산 결과를 재사용하여 생성 속도를 높입니다.
        )

    # 생성된 토큰 ID 전체를 문자열로 복원합니다.
    # skip_special_tokens=True는 </s>, <pad> 같은 특수 토큰을 출력에서 제외합니다.
    # clean_up_tokenization_spaces=False는 영어권 토크나이저용 공백 자동 정리를 끄는 설정입니다.
    # 한국어 출력에서는 불필요한 공백 정리가 오히려 어색한 결과를 만들 수 있으므로 False로 둡니다.
    decoded_text = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    # 깨진 문자와 불필요한 공백을 정리합니다.
    decoded_text = clean_generated_text(decoded_text)

    # 최종 생성 문장을 반환합니다.
    return decoded_text

## 8. 시작 문장과 토큰 확인

모델은 한국어 문장을 직접 이해하는 것이 아니라 토큰 ID를 입력으로 사용합니다.  
따라서 먼저 시작 문장이 어떤 정수 ID와 토큰으로 바뀌는지 확인합니다.

In [ ]:
# 의미가 잘 이어지도록 시작 문장을 조금 더 구체적으로 작성합니다.
# 막연한 문장보다 목적과 방향이 분명한 문장이 생성 품질에 유리합니다.
prompt = "딥러닝을 잘 하기 위해서는 기초 수학과 파이썬을 꾸준히 공부해야 한다. 특히"

# 시작 문장을 토큰 ID 리스트로 변환합니다.
input_ids_list = tokenizer.encode(prompt, add_special_tokens=False)

# 토큰 ID를 실제 토큰 문자열로 변환합니다.
input_tokens = tokenizer.convert_ids_to_tokens(input_ids_list)

# 시작 문장을 출력합니다.
print("시작 문장:")
print(prompt)

# 정수 ID 리스트를 출력합니다.
print("\n정수 ID 리스트:")
print(input_ids_list)

# 토큰 목록을 출력합니다.
print("\n토큰 목록:")
print(input_tokens)

## 9. 개선된 설정으로 문장 생성하기

아래 코드는 기존의 단순 `generate()`보다 안정적인 설정을 사용합니다.  
새로 생성할 길이를 제한하고, 자연스러운 후보 안에서 샘플링하므로 시작 문장과 의미가 더 잘 이어질 가능성이 높습니다.

In [ ]:
# 개선된 생성 함수를 사용하여 문장을 생성합니다.
generated_text = generate_korean_text(
    prompt=prompt,              # 시작 문장입니다.
    max_new_tokens=60,          # 입력 뒤에 새로 생성할 토큰 수입니다.
    temperature=0.70,           # 너무 튀는 문장을 줄이기 위해 비교적 낮게 설정합니다.
    top_p=0.90,                 # 누적 확률 90% 안의 후보에서 선택합니다.
    top_k=40,                   # 상위 40개 후보 안에서 선택합니다.
    repetition_penalty=1.15,    # 반복 표현을 줄입니다.
    no_repeat_ngram_size=3      # 같은 3토큰 구절 반복을 막습니다.
)

# 최종 생성 결과를 출력합니다.
print("생성된 문장:")
print(generated_text)

## 10. 문장 연결성을 높이는 프롬프트 예시

생성 문장이 시작 문장과 잘 이어지게 하려면 시작 문장을 구체적으로 작성하는 것이 중요합니다.

좋은 시작 문장의 특징은 다음과 같습니다.

- 주제가 명확합니다.
- 문장이 너무 짧지 않습니다.
- 이어질 방향을 암시합니다.
- 갑자기 끊긴 형태보다 자연스럽게 이어질 수 있는 형태가 좋습니다.

In [ ]:
# 여러 시작 문장을 준비합니다.
# 각 문장은 생성 결과가 어떤 식으로 달라지는지 비교하기 위한 예시입니다.
prompts = [
    "인공지능을 공부할 때 가장 먼저 이해해야 할 개념은 데이터와 모델의 관계이다. 데이터는",
    "자연어 처리를 잘 이해하기 위해서는 토큰화 과정을 먼저 알아야 한다. 토큰화는",
    "파이토치로 딥러닝 모델을 만들 때 중요한 흐름은 데이터 준비, 모델 정의, 학습, 평가이다. 이 중에서",
]

# 각 시작 문장에 대해 문장 생성을 수행합니다.
for i, p in enumerate(prompts, start=1):
    # 현재 예시 번호와 시작 문장을 출력합니다.
    print(f"\n===== 예시 {i} =====")
    print("시작 문장:", p)

    # 동일한 생성 함수를 사용하여 이어지는 문장을 만듭니다.
    result = generate_korean_text(
        prompt=p,
        max_new_tokens=50,
        temperature=0.70,
        top_p=0.90,
        top_k=40,
        repetition_penalty=1.15,
        no_repeat_ngram_size=3
    )

    # 생성 결과를 출력합니다.
    print("생성 결과:", result)

## 11. 다음 토큰 후보 점수 확인하기

GPT는 현재까지의 토큰을 기준으로 다음 토큰 후보들의 점수를 계산합니다.  
`logits`는 Softmax를 적용하기 전 점수이며, 값이 클수록 다음 토큰으로 선택될 가능성이 높습니다.

In [ ]:
# 시작 문장을 PyTorch Tensor 형태로 인코딩합니다.
encoded = tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False
)

# input_ids를 모델과 같은 device로 이동합니다.
input_ids = encoded["input_ids"].to(device)

# attention_mask를 모델과 같은 device로 이동합니다.
attention_mask = encoded["attention_mask"].to(device)

# 다음 토큰 후보 점수 계산은 추론이므로 기울기 계산을 비활성화합니다.
with torch.no_grad():
    # 모델에 현재까지의 입력 문장을 넣어 logits를 계산합니다.
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)

# logits는 [배치 크기, 입력 토큰 개수, 어휘 사전 크기] 형태입니다.
print("logits shape:", outputs.logits.shape)

## 12. 상위 5개 다음 토큰 후보 확인

마지막 토큰 위치의 `logits`를 확인하면 다음에 올 가능성이 높은 토큰 후보를 볼 수 있습니다.  
이 과정은 GPT가 문장을 생성할 때 내부적으로 어떤 후보를 고려하는지 이해하는 데 도움이 됩니다.

In [ ]:
# 첫 번째 문장의 마지막 위치에 해당하는 다음 토큰 후보 점수를 선택합니다.
last_token_logits = outputs.logits[0, -1, :]

# torch.topk()는 점수가 가장 높은 k개 값과 해당 토큰 ID를 반환합니다.
top5_values, top5_indices = torch.topk(last_token_logits, k=5)

# GPU Tensor를 CPU 리스트로 변환합니다.
top5_token_ids = top5_indices.cpu().tolist()

# 상위 토큰 ID를 실제 토큰 문자열로 변환합니다.
top5_tokens = tokenizer.convert_ids_to_tokens(top5_token_ids)

# 상위 후보를 순위별로 출력합니다.
print("상위 5개 다음 토큰 후보:")
for rank, (token_id, token, score) in enumerate(zip(top5_token_ids, top5_tokens, top5_values.cpu().tolist()), start=1):
    # 토큰 문자열에 깨진 문자가 있을 수 있으므로 보기 좋게 정리합니다.
    token_text = clean_generated_text(token)

    # 순위, 토큰 ID, 토큰, 점수를 출력합니다.
    print(f"{rank}위 | 토큰 ID: {token_id} | 토큰: {token_text} | 점수: {score:.4f}")

## 13. 생성 옵션 비교 실습

아래 코드는 안정적인 생성과 다양한 생성을 비교합니다.

- 안정적인 생성: `temperature`가 낮고 후보 범위가 좁아 문장이 비교적 차분합니다.
- 다양한 생성: `temperature`가 높고 후보 범위가 넓어 표현은 다양하지만 문맥이 흔들릴 수 있습니다.

In [ ]:
# 비교에 사용할 시작 문장입니다.
compare_prompt = "머신러닝 모델의 성능을 높이기 위해서는 데이터를 잘 전처리해야 한다. 예를 들어"

# 안정적인 생성 옵션을 적용합니다.
stable_result = generate_korean_text(
    prompt=compare_prompt,
    max_new_tokens=50,
    temperature=0.60,
    top_p=0.85,
    top_k=30,
    repetition_penalty=1.15,
    no_repeat_ngram_size=3
)

# 다양한 생성 옵션을 적용합니다.
creative_result = generate_korean_text(
    prompt=compare_prompt,
    max_new_tokens=50,
    temperature=0.95,
    top_p=0.95,
    top_k=80,
    repetition_penalty=1.10,
    no_repeat_ngram_size=3
)

# 비교 결과를 출력합니다.
print("시작 문장:")
print(compare_prompt)

print("\n[안정적인 생성 결과]")
print(stable_result)

print("\n[다양한 생성 결과]")
print(creative_result)